# Airfoil Self-Noise Prediction: run the full workflow

MMA3001 individual project. This notebook downloads the project code from
GitHub, gets the official NASA/UCI data, re-runs every result, runs the tests
and rebuilds the documentation.

**How to use:** Runtime → Run all. Takes about a minute.

Repository: https://github.com/ellrohde/Air-Foil-Noise

## 1. Set up
Clone the repository (first run only), move into it, and install the packages
Colab doesn't already have. `%cd` is the notebook "magic" command for changing
directory (Week 1); `!` runs a terminal command.

In [ ]:
# Always start from /content, so re-running this cell never puts the repo inside itself.
%cd /content
import os
if not os.path.exists("Air-Foil-Noise"):
    !git clone -q https://github.com/ellrohde/Air-Foil-Noise.git
%cd /content/Air-Foil-Noise
!git pull -q        # make sure this is the latest version on GitHub
!ls

In [ ]:
%cd /content/Air-Foil-Noise
# Install the packages Colab doesn't already have.
#   ucimlrepo   - downloads the NASA airfoil dataset from UCI
#   pytest-html - makes the HTML test report
#   pdoc        - builds the HTML code documentation
# (numpy, pandas, scikit-learn, matplotlib and pytest are already in Colab.)
!pip install -q ucimlrepo pytest-html pdoc

## 2. Run the workflow
`run_all.py` downloads the UCI data, audits it, runs the degree sweep with
random and grouped cross-validation, scores the final test runs, does the
physical and numerical checks, and saves `results/` and `figures/`.

In [ ]:
%cd /content/Air-Foil-Noise
!python run_all.py

## 3. Results summary
Read straight from the files `run_all.py` just wrote, so these always match
this run.

In [ ]:
%cd /content/Air-Foil-Noise
import json
import pandas as pd

print("Final test (whole wind-tunnel runs never used in development):")
display(pd.read_csv("results/final_test.csv").round(3))

print("Mean validation RMSE (dB) by degree: random vs grouped split")
sweep = pd.read_csv("results/degree_sweep.csv")
display(sweep.pivot(index="degree", columns="split", values="val_rmse").round(2))

print("Physical check: SPL increase from 31.7 to 71.3 m/s")
check = json.load(open("results/velocity_check.json"))
display(pd.DataFrame(check).T[["pairs", "model_increase_fraction",
                               "mean_measured_increase_db",
                               "mean_predicted_increase_db"]].round(2))

## 4. Figures

In [ ]:
%cd /content/Air-Foil-Noise
from IPython.display import Image

figures = [
    "degree_sweep.png",            # random vs grouped split, degree 1-5
    "runtime.png",                 # fit time and number of terms
    "observed_vs_predicted.png",   # final test: measured vs predicted SPL
    "residuals_vs_frequency.png",  # where the selected model is wrong
    "example_spectra.png",         # full noise spectra for 3 held-out runs
    "velocity_check.png",          # physical check: noise vs velocity
    "permutation_importance.png",  # sensitivity: which inputs matter most
]
for name in figures:
    print(name)
    display(Image(filename=f"figures/{name}", width=700))

## 5. Tests
Runs all pytest tests and writes the HTML test report to `reports/`.

In [ ]:
%cd /content/Air-Foil-Noise
!python -m pytest -v --html=reports/test_report.html --self-contained-html

## 6. Code documentation
Builds the HTML documentation from the docstrings into `docs/`.

In [ ]:
%cd /content/Air-Foil-Noise
!pdoc -d numpy -o docs ./src/airfoil_noise

## 7. Download the outputs
Copy the contents of this zip into the repository folder on your computer and
commit them.

In [ ]:
%cd /content/Air-Foil-Noise
!rm -f colab_outputs.zip
!zip -qr colab_outputs.zip results figures reports docs
from google.colab import files
files.download("colab_outputs.zip")